In [5]:
"""
Compare Order.Line values between two Excel tabs (SAS vs PBI)
Outputs discrepancies to a new tab in the same file

What this script does:
This script compares two lists of Order.Line numbers from your SAS and PBI systems to help you validate that both systems have the same data.
Here's the step-by-step process:

Reads your Excel file - Opens the file and loads data from both the "SAS" and "PBI" tabs
Normalizes the Order.Line numbers - This is the critical part! It converts all order numbers to a consistent text format with at least 2 decimal places. So:

6290220.5 becomes 6290220.50
6277472.01 stays 6277472.01
6277472.501 stays 6277472.501

This fixes the problem you had where Excel was treating "6290220.5" (text) differently from 6290220.5 (number).
Compares the two lists and categorizes every order into buckets:

In Both - Orders that exist in both systems ✓
Only in SAS - Orders missing from PBI ⚠️
Only in PBI - Orders missing from SAS ⚠️
Duplicated in SAS - Orders appearing multiple times in SAS ⚠️ (now with counts!)
Duplicated in PBI - Orders appearing multiple times in PBI ⚠️ (now with counts!)


Creates a new Excel file with a "Comparison Results" tab that shows all these categories in filterable columns

Why you need this:
You're validating two source systems (SAS vs PBI). They should have identical Order.Line data, but:

Some orders might only exist in one system (data sync issues)
Some orders might be duplicated (data quality issues)
The formatting differences make manual comparison impossible

This script automatically finds all the discrepancies so you can investigate why they're different and fix the underlying data issues.
"""

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
import sys

# ============= CONFIGURATION =============
INPUT_FILE = r"C:\Users\Chris Gilson\OneDrive\Documents\Dupuis Analytics\OmniMax\SAS vs PBI Order Line Comparison.xlsx"
SAS_TAB = "SAS"
PBI_TAB = "PBI"
COLUMN_NAME = "Order.Line"
OUTPUT_TAB = "Comparison Results"
# =========================================


def normalize_order_line(value):
    """Convert Order.Line to standardized text format with minimum 2 decimals"""
    if pd.isna(value):
        return None
    
    # Convert to string and strip whitespace
    value_str = str(value).strip()
    
    # Handle if it's already a float
    try:
        value_float = float(value_str)
        # Format with at least 2 decimals, preserving up to 3 if they exist
        if value_float == int(value_float):
            return f"{int(value_float)}.00"
        elif value_float * 10 == int(value_float * 10):
            return f"{value_float:.1f}0"
        elif value_float * 100 == int(value_float * 100):
            return f"{value_float:.2f}"
        else:
            return f"{value_float:.3f}"
    except:
        return value_str


def main():
    print(f"Reading file: {INPUT_FILE}")
    
    # Read both tabs
    try:
        sas_df = pd.read_excel(INPUT_FILE, sheet_name=SAS_TAB)
        pbi_df = pd.read_excel(INPUT_FILE, sheet_name=PBI_TAB)
    except Exception as e:
        print(f"Error reading file: {e}")
        sys.exit(1)
    
    # Check if column exists
    if COLUMN_NAME not in sas_df.columns:
        print(f"Error: Column '{COLUMN_NAME}' not found in {SAS_TAB} tab")
        print(f"Available columns: {list(sas_df.columns)}")
        sys.exit(1)
    
    if COLUMN_NAME not in pbi_df.columns:
        print(f"Error: Column '{COLUMN_NAME}' not found in {PBI_TAB} tab")
        print(f"Available columns: {list(pbi_df.columns)}")
        sys.exit(1)
    
    # Normalize Order.Line values
    sas_normalized = sas_df[COLUMN_NAME].apply(normalize_order_line).dropna()
    pbi_normalized = pbi_df[COLUMN_NAME].apply(normalize_order_line).dropna()
    
    sas_orders = set(sas_normalized)
    pbi_orders = set(pbi_normalized)
    
    # Find duplicates within each tab with counts
    sas_counts = sas_normalized.value_counts()
    pbi_counts = pbi_normalized.value_counts()
    
    sas_duplicates = sorted([x for x in sas_orders if sas_counts[x] > 1])
    pbi_duplicates = sorted([x for x in pbi_orders if pbi_counts[x] > 1])
    
    # Create duplicate lists with counts
    sas_duplicates_with_counts = [(order, sas_counts[order]) for order in sas_duplicates]
    pbi_duplicates_with_counts = [(order, pbi_counts[order]) for order in pbi_duplicates]
    
    # Find discrepancies
    only_in_sas = sorted(sas_orders - pbi_orders)
    only_in_pbi = sorted(pbi_orders - sas_orders)
    in_both = sorted(sas_orders & pbi_orders)
    
    # Generate SQL queries
    def generate_sql_query(orders, query_name):
        if not orders:
            return f"-- No orders found for {query_name}"
        
        sql = f"""-- Query to investigate {len(orders)} orders {query_name}
-- Generated from comparison results

SELECT db.branch_name_short
    , fosd.actual_ship_date
    , fosd.line_promise_date
    , fosd.line_request_date
    , fosd.ship_line_count
    , fosd.company
    , fosd.cpu
    , fosd.market
    , fosd.order_quantity
    , fosd.ship_quantity
    , fosd.first_ship_date
    , fosd.first_ship_quantity
    , fosd.int_ext
    , fosd.line_status
    , fosd.* 
FROM gold.fact_order_ship_detail fosd
LEFT JOIN gold.dim_branches db ON db.branch_id = fosd.ship_branch_id
WHERE order_line_cmb IN (
"""
        order_strings = [f"    '{order}'" for order in orders]
        sql += ",\n".join(order_strings)
        sql += "\n);"
        return sql
    
    sql_only_in_pbi = generate_sql_query(only_in_pbi, "in PBI but not SAS")
    sql_only_in_sas = generate_sql_query(only_in_sas, "in SAS but not PBI")
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"COMPARISON SUMMARY")
    print(f"{'='*60}")
    print(f"SAS total records: {len(sas_normalized):,} ({len(sas_duplicates):,} duplicates)")
    print(f"PBI total records: {len(pbi_normalized):,} ({len(pbi_duplicates):,} duplicates)")
    print(f"Unique in both: {len(in_both):,}")
    print(f"Unique only in SAS: {len(only_in_sas):,}")
    print(f"Unique only in PBI: {len(only_in_pbi):,}")
    print(f"{'='*60}\n")
    
    # Create results dataframe with duplicate counts
    max_len = max(
        len(only_in_sas), 
        len(only_in_pbi), 
        len(in_both), 
        len(sas_duplicates_with_counts), 
        len(pbi_duplicates_with_counts)
    )
    
    # Separate Order.Line and Count for duplicates
    sas_dup_orders = [order for order, count in sas_duplicates_with_counts]
    sas_dup_counts = [count for order, count in sas_duplicates_with_counts]
    
    pbi_dup_orders = [order for order, count in pbi_duplicates_with_counts]
    pbi_dup_counts = [count for order, count in pbi_duplicates_with_counts]
    
    results_df = pd.DataFrame({
        'In Both': in_both + [''] * (max_len - len(in_both)),
        'Only in SAS': only_in_sas + [''] * (max_len - len(only_in_sas)),
        'Only in PBI': only_in_pbi + [''] * (max_len - len(only_in_pbi)),
        'Duplicated in SAS': sas_dup_orders + [''] * (max_len - len(sas_dup_orders)),
        'SAS Dup Count': sas_dup_counts + [''] * (max_len - len(sas_dup_counts)),
        'Duplicated in PBI': pbi_dup_orders + [''] * (max_len - len(pbi_dup_orders)),
        'PBI Dup Count': pbi_dup_counts + [''] * (max_len - len(pbi_dup_counts))
    })
    
    # Generate output filename
    output_file = INPUT_FILE.replace('.xlsx', '_result.xlsx')
    
    # Load workbook with openpyxl to preserve existing sheets
    try:
        wb = load_workbook(INPUT_FILE)
        
        # Remove output tab if it already exists
        if OUTPUT_TAB in wb.sheetnames:
            del wb[OUTPUT_TAB]
        
        # Create new sheet
        ws = wb.create_sheet(OUTPUT_TAB)
        
        # Add headers
        ws['A1'] = 'In Both'
        ws['B1'] = 'Only in SAS'
        ws['C1'] = 'Only in PBI'
        ws['D1'] = 'Duplicated in SAS'
        ws['E1'] = 'SAS Dup Count'
        ws['F1'] = 'Duplicated in PBI'
        ws['G1'] = 'PBI Dup Count'
        
        # Style headers
        header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
        header_font = Font(bold=True, color='FFFFFF')
        
        for col in ['A', 'B', 'C', 'D', 'E', 'F', 'G']:
            ws[f'{col}1'].fill = header_fill
            ws[f'{col}1'].font = header_font
            ws[f'{col}1'].alignment = Alignment(horizontal='center')
        
        # Add data
        for idx, row in results_df.iterrows():
            ws[f'A{idx+2}'] = row['In Both']
            ws[f'B{idx+2}'] = row['Only in SAS']
            ws[f'C{idx+2}'] = row['Only in PBI']
            ws[f'D{idx+2}'] = row['Duplicated in SAS']
            ws[f'E{idx+2}'] = row['SAS Dup Count']
            ws[f'F{idx+2}'] = row['Duplicated in PBI']
            ws[f'G{idx+2}'] = row['PBI Dup Count']
        
        # Highlight discrepancy columns
        highlight_fill = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')
        duplicate_fill = PatternFill(start_color='FFE6E6', end_color='FFE6E6', fill_type='solid')
        count_fill = PatternFill(start_color='E6F3FF', end_color='E6F3FF', fill_type='solid')
        
        for row in range(2, len(only_in_sas) + 2):
            ws[f'B{row}'].fill = highlight_fill
        for row in range(2, len(only_in_pbi) + 2):
            ws[f'C{row}'].fill = highlight_fill
        for row in range(2, len(sas_dup_orders) + 2):
            ws[f'D{row}'].fill = duplicate_fill
            ws[f'E{row}'].fill = count_fill  # Light blue for counts
        for row in range(2, len(pbi_dup_orders) + 2):
            ws[f'F{row}'].fill = duplicate_fill
            ws[f'G{row}'].fill = count_fill  # Light blue for counts
        
        # Set column widths
        ws.column_dimensions['A'].width = 20
        ws.column_dimensions['B'].width = 20
        ws.column_dimensions['C'].width = 20
        ws.column_dimensions['D'].width = 20
        ws.column_dimensions['E'].width = 15  # Count column
        ws.column_dimensions['F'].width = 20
        ws.column_dimensions['G'].width = 15  # Count column
        
        # Add SQL queries section
        sql_start_row = max_len + 6  # Leave some space after data
        
        ws[f'A{sql_start_row}'] = 'SQL QUERIES - COPY/PASTE TO INVESTIGATE'
        ws[f'A{sql_start_row}'].font = Font(bold=True, size=14)
        ws.merge_cells(f'A{sql_start_row}:G{sql_start_row}')
        
        # Query for PBI only
        ws[f'A{sql_start_row + 2}'] = f'Query for "Only in PBI" ({len(only_in_pbi)} orders):'
        ws[f'A{sql_start_row + 2}'].font = Font(bold=True, size=12)
        ws.merge_cells(f'A{sql_start_row + 2}:G{sql_start_row + 2}')
        
        ws[f'A{sql_start_row + 3}'] = sql_only_in_pbi
        ws[f'A{sql_start_row + 3}'].alignment = Alignment(wrap_text=True, vertical='top')
        ws.merge_cells(f'A{sql_start_row + 3}:G{sql_start_row + 3}')
        ws.row_dimensions[sql_start_row + 3].height = 300
        
        # Query for SAS only
        ws[f'A{sql_start_row + 5}'] = f'Query for "Only in SAS" ({len(only_in_sas)} orders):'
        ws[f'A{sql_start_row + 5}'].font = Font(bold=True, size=12)
        ws.merge_cells(f'A{sql_start_row + 5}:G{sql_start_row + 5}')
        
        ws[f'A{sql_start_row + 6}'] = sql_only_in_sas
        ws[f'A{sql_start_row + 6}'].alignment = Alignment(wrap_text=True, vertical='top')
        ws.merge_cells(f'A{sql_start_row + 6}:G{sql_start_row + 6}')
        ws.row_dimensions[sql_start_row + 6].height = 300
        
        # Add summary at top
        ws.insert_rows(1, 3)
        ws['A1'] = 'COMPARISON SUMMARY'
        ws['A1'].font = Font(bold=True, size=14)
        ws['A2'] = f'SAS: {len(sas_normalized):,} records ({len(sas_duplicates):,} dups) | PBI: {len(pbi_normalized):,} records ({len(pbi_duplicates):,} dups) | Both: {len(in_both):,} | SAS Only: {len(only_in_sas):,} | PBI Only: {len(only_in_pbi):,}'
        ws.merge_cells('A2:G2')
        
        # Save
        wb.save(output_file)
        print(f"Results saved to: {output_file}")
        print(f"New tab '{OUTPUT_TAB}' added with comparison results")
        
    except Exception as e:
        print(f"Error writing output: {e}")
        sys.exit(1)


if __name__ == "__main__":
    main()

Reading file: C:\Users\Chris Gilson\OneDrive\Documents\Dupuis Analytics\OmniMax\SAS vs PBI Order Line Comparison.xlsx

COMPARISON SUMMARY
SAS total records: 5,696 (1 duplicates)
PBI total records: 6,009 (1 duplicates)
Unique in both: 5,695
Unique only in SAS: 0
Unique only in PBI: 313

Results saved to: C:\Users\Chris Gilson\OneDrive\Documents\Dupuis Analytics\OmniMax\SAS vs PBI Order Line Comparison_result.xlsx
New tab 'Comparison Results' added with comparison results
